In [ ]:
from h5flow.data import dereference
import h5py
import numpy as np
import numpy.lib.recfunctions as rfn
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform, cdist
from sklearn.linear_model import LinearRegression
import uproot

In [ ]:
# from asymmetry_prep import yz_line, proj_yz, xline

In [ ]:
def line1D(X, y):
    reg = LinearRegression().fit(X.reshape(-1, 1), y)
    return reg.coef_[0], reg.intercept_

def yz_line(hits):
    '''
    z = ky + b
    dz = kdy
    return k, b, (1/sqrt(1+k**2), k/sqrt(1+k**2))
    tagent vector = (1/sqrt(1+k**2), k/sqrt(1+k**2))
    '''
    # reg = LinearRegression().fit(hits['y'].reshape(-1, 1), hits['z'])
    # k = reg.coef_[0]
    # b = reg.intercept_
    k, b = line1D(hits['y'], hits['z'])
    return k, b, (1/np.sqrt(1+k**2), k/np.sqrt(1+k**2))

def proj_yz(hits, k_yz, b_yz):
    '''
    z = ky + b
    '''
    tg_yz = np.array([1/np.sqrt(1+k_yz**2), k_yz/np.sqrt(1+k_yz**2)])
    o_yz = (0, b_yz)
    points_yz = np.column_stack([hits['y'], hits['z']]) - o_yz
    d_tg = np.dot(points_yz, tg_yz)
    # for i in range(10):
    #     print(tg_yz[0]* points_yz[i][0] + tg_yz[1]* points_yz[i][1] - d_tg[i], points_yz[i], rock_muon_hits['y'][i], rock_muon_hits['z'][i], o_yz)
    d_nm = np.cross(points_yz, tg_yz)
    # for i in range(10):
    #     print(-tg_yz[0]* points_yz[i][1] + tg_yz[1]* points_yz[i][0] - d_nm[i], points_yz[i], rock_muon_hits['y'][i], rock_muon_hits['z'][i], o_yz, tg_yz)
    return d_tg, d_nm

def xline(hits, reflabel='d_tg'):
    '''
    x = k* reflabel + b
    '''
    k, b = line1D(hits[reflabel], hits['x'])
    return k, b, (1/np.sqrt(1+k**2), k/np.sqrt(1+k**2))

In [ ]:
def prepare_tracks(f_name):
    f = h5py.File(f_name, 'r')
    tracks2hits = dereference(
        f['/analysis/rock_muon_tracks/data']['rock_muon_id'],     # indices of A to load references for, shape: (n,)
        f['/analysis/rock_muon_tracks/ref/charge/calib_prompt_hits/ref'],  # references to use, shape: (L,)
        f['/charge/calib_prompt_hits/data'],
        ref_direction = (0,1)# dataset to load, shape: (M,)
    )
    tracks = f['/analysis/rock_muon_tracks/data']
    
    return tracks, tracks2hits

In [ ]:
def t2h(tracks, tracks2hits):
    angle_mask = np.abs(tracks['x_start'] - tracks['x_end'])/tracks['length'] < 0.05
    trks = tracks[angle_mask]
    t2hs = tracks2hits[angle_mask]
    rock_muon_hits = []
    track_ids = []
    for i in range(len(trks)):
        track = trks[i]
        t2h = t2hs[i]
        rock_muon_hits.append(np.array([tup for tup in t2h if not any(tup.mask)], dtype = t2h.dtype))
        track_ids.append(np.full(len(rock_muon_hits[-1]), fill_value=i, dtype=int))
    rock_muon_hits = np.concatenate(rock_muon_hits)
    track_ids = np.concatenate(track_ids)
    rock_muon_hits = rfn.append_fields(rock_muon_hits, names=['event_id', ], data=[track_ids, ], usemask=False)
    return rock_muon_hits

In [ ]:
def sel_uni_pxl(hits, att='Q', yposlabel='y', zposlabel='z'):
    points_yz = np.column_stack([hits['io_group'].astype(float), hits['io_channel'].astype(float), hits[yposlabel], hits[zposlabel]])

    clustering = DBSCAN(eps=0.000001, min_samples=1).fit(points_yz)
    selected_hit = []
    unique_labels = np.unique(clustering.labels_)
    totQ = np.zeros(hits.shape[0], dtype=float)
    totN = np.zeros(hits.shape[0], dtype=int)
    totQ_cp = totQ.copy()
    totN_cp = totN.copy()
    accQ = np.zeros(hits.shape[0], dtype=float)
    avg_i = np.zeros(hits.shape[0], dtype=float)
    dt = np.zeros(hits.shape[0], dtype=float)
    tinterval = np.zeros(hits.shape[0], dtype=float)
    tindex = np.zeros(hits.shape[0], dtype=float)

    # dist = pdist((points_yz - clustering.components_).reshape(-1, 1))
    # if np.sum(dist)>1E-6:
    #     print(np.sum(dist))
    # print(np.sum(dist))
   

    for ilabel, unique_label in enumerate(unique_labels):
        m = clustering.labels_ == unique_label
        idxs = np.asarray(m).nonzero()[0]
        d = hits[m]
        selected_hit.append(d[np.argmax(d[att])])
        totQ[ilabel] = np.sum(d['Q'])
        totN[ilabel] = len(d)
        totQ_cp[m] = np.sum(d['Q'])
        totN_cp[m] = len(d)
        sorted_indices = np.argsort(d['t_drift'])
        dt[m] = hits['t_drift'][m] - selected_hit[-1]['t_drift']
        q = 0
        for i in range(len(sorted_indices)):
            q += d['Q'][sorted_indices[i]]
            accQ[idxs[sorted_indices[i]]] = np.float64(q)
            if i == 0:
                avg_i[idxs[sorted_indices[i]]] = (d['Q'][sorted_indices[i]]-5)/17
                tinterval[idxs[sorted_indices[i]]] = 17
            else:
                avg_i[idxs[sorted_indices[i]]] = d['Q'][sorted_indices[i]] / (d['t_drift'][sorted_indices[i]] - d['t_drift'][sorted_indices[i-1]])
                tinterval[idxs[sorted_indices[i]]] = d['t_drift'][sorted_indices[i]] - d['t_drift'][sorted_indices[i-1]]
                if tinterval[idxs[sorted_indices[i]]] < 1:
                    # print(hits[idxs[sorted_indices]])
                    print(hits[idxs[sorted_indices[i]]]['y'] - hits[idxs[sorted_indices[i-1]]]['y'])
            tindex[idxs[sorted_indices[i]]] = i
    uni_pxl = np.array(selected_hit, dtype=hits.dtype)
    totQ = totQ[:len(uni_pxl)]
    totN = totN[:len(uni_pxl)]
    uni_pxl = rfn.append_fields(uni_pxl, names=['totQ', 'totN'], data=[totQ, totN], usemask=False)
    extended_hits = rfn.append_fields(hits, names=['totQ', 'totN'], data=[totQ_cp, totN_cp], usemask=False)
    extended_hits = rfn.append_fields(extended_hits, names=['accQ', 'dt', 'avg_i'], data=[accQ, dt, avg_i], usemask=False)
    extended_hits = rfn.append_fields(extended_hits, names=['tinterval', 'tindex'], data=[tinterval, tindex], usemask=False)
    return uni_pxl, extended_hits

In [ ]:
tracks, tracks2hits = prepare_tracks('/home/yousen/Public/ndlar_shared/data/packet-0050017-2024_07_08_15_13_35_CDT.FLOW.rock_mu.h5')

In [ ]:
rock_muon_hits = t2h(tracks, tracks2hits)

In [ ]:
rock_muon_hits.dtype

In [ ]:
def plot_three_views(hits, axs=None, **kwargs):
    # ridx, cidx
    axis_dict = {
        # y vs. x
        (0, 0) : { 'label' : {'x' : 'x [cm]', 'y' : 'y [cm]'},
                    'key' : {'x' : 'x', 'y' : 'y'}},
        # (0, 1) : # empty
        (1, 0) : { 'label' : {'x' : 'x [cm]', 'y' : 'z [cm]'},
                  'key' : {'x' : 'x', 'y' : 'z'}},
        (1, 1) : { 'label' : {'x' : 'y [cm]', 'y' : 'z [cm]'},
                  'key' : {'x' : 'y', 'y' : 'z'}}
    }
    all_hits = kwargs.get('all_hits', None)
    if isinstance(all_hits, np.ndarray):
        for k, v in axis_dict.items():
            axs[k[0], k[1]].scatter(all_hits[v['key']['x']], all_hits[v['key']['y']], label='all hits')
    for k, v in axis_dict.items():
        axs[k[0], k[1]].scatter(hits[v['key']['x']], hits[v['key']['y']], label=kwargs.get('label', 'selected'))

    for k, v in axis_dict.items():
        axs[k[0], k[1]].set_xlabel(v['label']['x'])
        axs[k[0], k[1]].set_ylabel(v['label']['y'])
        axs[k[0], k[1]].legend()

In [ ]:
def prep_per_event(hits, highq_thres=None):
    uni_pxls, extended_hits = sel_uni_pxl(hits, att='t_drift', yposlabel='y', zposlabel='z')

    k_yz, b_yz, tg_yz = yz_line(uni_pxls)
    d_tg, d_nm = proj_yz(extended_hits, k_yz, b_yz)
    extended_hits = rfn.append_fields(extended_hits, names=('d_tg', 'd_nm'), data=(d_tg, d_nm), usemask=False)

    if highq_thres is not None:
        xyhits = extended_hits[extended_hits['Q'] > highq_thres]
    else:
        xyhits = uni_pxls
    k_x, b_x, _ = xline(xyhits, reflabel='y')
    ref_x = k_x * extended_hits['y'] + b_x
    dx = extended_hits['x'] - ref_x
    sign = np.where(extended_hits['io_group'] % 2, 1.0, -1.0)
    dx *= sign
    extended_hits = rfn.append_fields(extended_hits, names='dx', data=dx, usemask=False)
    return extended_hits


In [ ]:
trkid = np.unique(rock_muon_hits['event_id'])

In [ ]:
import matplotlib.pyplot as plt
out_hits = []
for i in trkid:
    
    hits = rock_muon_hits[rock_muon_hits['event_id'] == i]
    for j in [1,2,3,4,7,8]:
        muhits = hits[hits['io_group'] == j]

        if len(muhits) < 5:
            continue
        if i == 14:
            continue
        fig, axs = plt.subplots(2,2, figsize=(10,10))
        plot_three_views(muhits, axs)
        axs[0,0].set_title(f'trk id {i}; io group {j}')
        fig.savefig(f'from_data/trk{i}_iogroup{j}.png')
        out_hits.append(prep_per_event(muhits))

In [ ]:
with uproot.recreate('rock_mu.root') as froot:
    froot['rockmu'] = np.concatenate(out_hits)